# State-Dependent U.S. Equity Sector Rotation
## A Systematic Framework for Trend-Based Active Sector Allocation

# Block 2 — Universe, Data & Strategic Weight Methodology

This block resolves the empirical data architecture before any strategy signals are constructed.

## Canonical strategic allocation

The primary strategic allocation is now:

**52-week inverse-volatility sector weights**

This replaces the earlier point-in-time S&P 500 sector-weight target as the canonical baseline. Historical S&P 500 sector weights may still be examined later as a robustness extension, but they are no longer part of the core Block 2 workflow.

This keeps the primary framework fully reproducible from public price data and ensures that both active strategies are evaluated on the same strategic base.

## Timing convention inherited from Block 1

**last available U.S. close in the Friday-labelled signal week → calculate signals/targets → next U.S. trading session, normally Monday → rebalance**

In ordinary weeks, the signal observation is Friday's close. If Friday is a U.S. market holiday, the signal uses the final available U.S. trading session in that Friday-labelled week. Execution occurs on the next available U.S. trading session.

This block therefore distinguishes:

- **signal week label** — `W-FRI`;
- **actual signal observation date** — final trading session in that week;
- **execution date** — next available U.S. trading session after the signal observation.

No Friday-close signal is allowed to affect holdings before the next executable U.S. session.


In [ ]:
# ============================================================
# BLOCK 2.1 — ENVIRONMENT, DRIVE & BLOCK 1 MANIFEST
# ============================================================

from pathlib import Path
from google.colab import drive
import json
import sys
import subprocess
import importlib.util

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation")

DIRS = {
    "root": PROJECT_ROOT,
    "data_raw": PROJECT_ROOT / "data" / "raw",
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "outputs": PROJECT_ROOT / "outputs",
    "figures": PROJECT_ROOT / "outputs" / "figures",
    "tables": PROJECT_ROOT / "outputs" / "tables",
    "manifests": PROJECT_ROOT / "manifests",
    "logs": PROJECT_ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

BLOCK1_MANIFEST = DIRS["manifests"] / "block_1_research_configuration.json"

if not BLOCK1_MANIFEST.exists():
    raise FileNotFoundError(
        "Block 1 manifest not found. Run Block 1 first.\n"
        f"Expected: {BLOCK1_MANIFEST}"
    )

with open(BLOCK1_MANIFEST, "r", encoding="utf-8") as f:
    block1 = json.load(f)

CONFIG = block1["research_config"]
SECTOR_RECORDS = block1["sector_universe"]
BENCHMARK_TICKER = block1["benchmark_ticker"]

# Enforce the corrected timing contract.
assert CONFIG["signal_frequency"] == "W-FRI"
assert CONFIG["signal_observation"] == "FRIDAY_CLOSE"
assert CONFIG["rebalance_frequency"] == "WEEKLY"
assert CONFIG["rebalance_day"] == "MONDAY"
assert CONFIG["rebalance_execution_rule"] == "NEXT_US_TRADING_SESSION_AFTER_SIGNAL"
assert int(CONFIG["minimum_signal_execution_lag_sessions"]) >= 1

print("Loaded and validated Block 1 timing contract.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Benchmark: {BENCHMARK_TICKER}")
print(f"Sector count: {len(SECTOR_RECORDS)}")


Mounted at /content/drive
Loaded and validated Block 1 timing contract.
Project root: /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation
Benchmark: SPY
Sector count: 11


In [ ]:
# ============================================================
# BLOCK 2.2 — PACKAGE SETUP & IMPORTS
# ============================================================

if importlib.util.find_spec("yfinance") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "yfinance>=0.2.65"]
    )

import numpy as np
import pandas as pd
import yfinance as yf

from datetime import datetime, timezone

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

print("yfinance:", yf.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


yfinance: 0.2.66
pandas: 2.2.3
numpy: 2.1.3


In [ ]:
# ============================================================
# BLOCK 2.3 — UNIVERSE RECONSTRUCTION
# ============================================================

SECTOR_UNIVERSE = (
    pd.DataFrame(SECTOR_RECORDS)
    .set_index("ticker")
)

SECTOR_TICKERS = SECTOR_UNIVERSE.index.tolist()
ALL_TICKERS = [BENCHMARK_TICKER] + SECTOR_TICKERS

assert len(SECTOR_TICKERS) == 11
assert len(set(ALL_TICKERS)) == 12

display(SECTOR_UNIVERSE)
print("Downloaded instruments:", ALL_TICKERS)


,sector
ticker,
XLC,Communication Services
XLY,Consumer Discretionary
XLP,Consumer Staples
XLE,Energy
XLF,Financials
XLV,Health Care
XLI,Industrials
XLK,Information Technology
XLB,Materials


Downloaded instruments: ['SPY', 'XLC', 'XLY', 'XLP', 'XLE', 'XLF', 'XLV', 'XLI', 'XLK', 'XLB', 'XLRE', 'XLU']


## Historical-universe constraint

The modern 11-sector ETF panel has unequal inception dates. XLC and XLRE have shorter histories than the older Select Sector funds.

The canonical 11-sector sample therefore begins only when **all 11 ETFs have real observations**. Block 2 determines that date from the downloaded data instead of synthesizing unavailable pre-inception prices.


In [ ]:
# ============================================================
# BLOCK 2.4 — DOWNLOAD DAILY ADJUSTED MARKET DATA
# ============================================================

DOWNLOAD_START = "1998-01-01"
DOWNLOAD_END = None

raw = yf.download(
    tickers=ALL_TICKERS,
    start=DOWNLOAD_START,
    end=DOWNLOAD_END,
    interval="1d",
    auto_adjust=True,
    actions=False,
    repair=True,
    group_by="column",
    threads=True,
    progress=False,
)

if raw.empty:
    raise RuntimeError("No market data were downloaded.")

if not isinstance(raw.columns, pd.MultiIndex):
    raise RuntimeError("Unexpected yfinance column format.")

available_fields = raw.columns.get_level_values(0).unique().tolist()
required_fields = {"Open", "High", "Low", "Close", "Volume"}

missing_fields = required_fields.difference(available_fields)
if missing_fields:
    raise RuntimeError(f"Missing required fields: {sorted(missing_fields)}")

raw = raw.sort_index()
raw.index = pd.to_datetime(raw.index).tz_localize(None)

print(f"Downloaded {len(raw):,} daily rows.")
print(f"Raw date range: {raw.index.min().date()} to {raw.index.max().date()}")


Downloaded 7,208 daily rows.
Raw date range: 1998-01-02 to 2026-08-28


In [ ]:
# ============================================================
# BLOCK 2.5 — EXTRACT PRICES & AUDIT COVERAGE
# ============================================================

daily_close = raw["Close"].copy().reindex(columns=ALL_TICKERS)
daily_open = raw["Open"].copy().reindex(columns=ALL_TICKERS)

coverage_rows = []

for ticker in ALL_TICKERS:
    s = daily_close[ticker].dropna()
    coverage_rows.append(
        {
            "ticker": ticker,
            "sector": (
                "S&P 500"
                if ticker == BENCHMARK_TICKER
                else SECTOR_UNIVERSE.loc[ticker, "sector"]
            ),
            "first_valid": s.index.min() if len(s) else pd.NaT,
            "last_valid": s.index.max() if len(s) else pd.NaT,
            "daily_obs": len(s),
            "missing_pct_full_download_frame": (
                100 * daily_close[ticker].isna().mean()
            ),
        }
    )

coverage = pd.DataFrame(coverage_rows).set_index("ticker")
display(coverage)

sector_first_valid = coverage.loc[SECTOR_TICKERS, "first_valid"]
sector_last_valid = coverage.loc[SECTOR_TICKERS, "last_valid"]

COMMON_DAILY_START = sector_first_valid.max()
COMMON_DAILY_END = sector_last_valid.min()

print("\nModern 11-sector common daily window:")
print(COMMON_DAILY_START.date(), "to", COMMON_DAILY_END.date())


,sector,first_valid,last_valid,daily_obs,missing_pct_full_download_frame
ticker,,,,,
SPY,S&P 500,1998-01-02,2026-08-28,7208,0.000000
XLC,Communication Services,2018-06-19,2026-08-28,2060,71.420644
XLY,Consumer Discretionary,1998-12-22,2026-08-28,6963,3.399001
XLP,Consumer Staples,1998-12-22,2026-08-28,6963,3.399001
XLE,Energy,1998-12-22,2026-08-28,6963,3.399001
XLF,Financials,1998-12-22,2026-08-28,6963,3.399001
XLV,Health Care,1998-12-22,2026-08-28,6963,3.399001
XLI,Industrials,1998-12-22,2026-08-28,6963,3.399001
XLK,Information Technology,1998-12-22,2026-08-28,6963,3.399001



Modern 11-sector common daily window:
2018-06-19 to 2026-08-28


In [ ]:
# ============================================================
# BLOCK 2.6 — DAILY DATA-QUALITY CHECKS
# ============================================================

common_daily_close = daily_close.loc[
    COMMON_DAILY_START:COMMON_DAILY_END,
    ALL_TICKERS
].copy()

common_daily_open = daily_open.loc[
    COMMON_DAILY_START:COMMON_DAILY_END,
    ALL_TICKERS
].copy()

non_positive_close = (common_daily_close <= 0).sum()
non_positive_open = (common_daily_open <= 0).sum()

if non_positive_close.sum() > 0:
    raise ValueError(
        "Non-positive adjusted close values detected:\n"
        + non_positive_close[non_positive_close > 0].to_string()
    )

if non_positive_open.sum() > 0:
    raise ValueError(
        "Non-positive adjusted open values detected:\n"
        + non_positive_open[non_positive_open > 0].to_string()
    )

missing_counts = common_daily_close.isna().sum().sort_values(ascending=False)
display(missing_counts.rename("missing_daily_close_observations").to_frame())

daily_returns = common_daily_close.pct_change(fill_method=None)
extreme_mask = daily_returns.abs() > 0.25

extreme_events = (
    daily_returns.where(extreme_mask)
    .stack()
    .rename("daily_return")
    .reset_index()
    .rename(columns={"level_1": "ticker"})
)

print(f"Extreme |daily return| > 25% observations: {len(extreme_events)}")
if len(extreme_events):
    display(extreme_events.head(20))

print("Daily data-quality checks complete.")


,missing_daily_close_observations
Ticker,
SPY,0
XLC,0
XLY,0
XLP,0
XLE,0
XLF,0
XLV,0
XLI,0
XLK,0


Extreme |daily return| > 25% observations: 0
Daily data-quality checks complete.


In [ ]:
# ============================================================
# BLOCK 2.7 — BUILD FRIDAY-LABELLED WEEKLY SIGNAL PANEL
# ============================================================

weekly_close = common_daily_close.resample("W-FRI").last()
weekly_close = weekly_close.dropna(how="any")

weekly_returns = weekly_close.pct_change(fill_method=None)

if weekly_close.empty:
    raise RuntimeError("Weekly panel is empty after alignment.")

print(f"Weekly signal observations: {len(weekly_close):,}")
print(
    "Friday-labelled weekly range:",
    weekly_close.index.min().date(),
    "to",
    weekly_close.index.max().date(),
)

display(weekly_close.head())


Weekly signal observations: 428
Friday-labelled weekly range: 2018-06-22 to 2026-08-28


Ticker,SPY,XLC,XLY,XLP,XLE,XLF,XLV,XLI,XLK,XLB,XLRE,XLU
Date,,,,,,,,,,,,
2018-06-22,243.248856,46.694256,51.599922,41.645058,26.518709,23.260498,74.078949,63.552570,32.848316,24.897751,24.642368,19.779367
2018-06-29,240.185333,45.806431,50.663586,41.564396,26.797472,22.848049,72.796768,62.694801,32.231220,24.706287,24.947439,20.250957
2018-07-06,243.850845,46.953220,51.127117,42.120960,26.702204,22.916786,75.020966,63.158710,32.968933,24.867962,25.382162,20.722534
2018-07-13,247.542831,47.804047,52.188606,42.540398,26.910397,23.183165,76.233353,64.550331,33.655579,24.923277,25.168612,20.477001
2018-07-20,247.622498,47.045696,51.984642,42.637192,26.426956,23.681538,75.675133,65.154305,33.660233,24.799894,24.787273,20.379566


In [ ]:
# ============================================================
# BLOCK 2.8 — SIGNAL OBSERVATION & EXECUTION CALENDAR
# ============================================================

us_trading_dates = pd.DatetimeIndex(
    common_daily_close[BENCHMARK_TICKER].dropna().index.unique()
).sort_values()

timing_rows = []

for week_label in weekly_close.index:
    week_start = week_label - pd.Timedelta(days=6)

    sessions_this_week = us_trading_dates[
        (us_trading_dates >= week_start)
        & (us_trading_dates <= week_label)
    ]

    if len(sessions_this_week) == 0:
        continue

    signal_date = sessions_this_week[-1]

    future_sessions = us_trading_dates[
        us_trading_dates > signal_date
    ]

    execution_date = (
        future_sessions[0]
        if len(future_sessions)
        else pd.NaT
    )

    timing_rows.append(
        {
            "signal_week_label": week_label,
            "signal_observation_date": signal_date,
            "signal_observation_weekday": signal_date.day_name(),
            "execution_date": execution_date,
            "execution_weekday": (
                execution_date.day_name()
                if pd.notna(execution_date)
                else None
            ),
            "calendar_days_signal_to_execution": (
                (execution_date - signal_date).days
                if pd.notna(execution_date)
                else np.nan
            ),
        }
    )

weekly_timing = (
    pd.DataFrame(timing_rows)
    .set_index("signal_week_label")
    .sort_index()
    .reindex(weekly_close.index)
)

executable_timing = weekly_timing.dropna(
    subset=["execution_date"]
).copy()

assert (
    executable_timing["execution_date"]
    > executable_timing["signal_observation_date"]
).all()

print("Weekly signal/execution mapping created.")
display(executable_timing.head(12))

print("\nExecution weekdays:")
display(
    executable_timing["execution_weekday"]
    .value_counts()
    .rename("weeks")
    .to_frame()
)


Weekly signal/execution mapping created.


,signal_observation_date,signal_observation_weekday,execution_date,execution_weekday,calendar_days_signal_to_execution
Date,,,,,
2018-06-22,2018-06-22,Friday,2018-06-25,Monday,3.0
2018-06-29,2018-06-29,Friday,2018-07-02,Monday,3.0
2018-07-06,2018-07-06,Friday,2018-07-09,Monday,3.0
2018-07-13,2018-07-13,Friday,2018-07-16,Monday,3.0
2018-07-20,2018-07-20,Friday,2018-07-23,Monday,3.0
2018-07-27,2018-07-27,Friday,2018-07-30,Monday,3.0
2018-08-03,2018-08-03,Friday,2018-08-06,Monday,3.0
2018-08-10,2018-08-10,Friday,2018-08-13,Monday,3.0
2018-08-17,2018-08-17,Friday,2018-08-20,Monday,3.0



Execution weekdays:


,weeks
execution_weekday,
Monday,387
Tuesday,40


In [ ]:
# ============================================================
# BLOCK 2.9 — 52-WEEK INVERSE-VOLATILITY STRATEGIC WEIGHTS
# ============================================================

INV_VOL_LOOKBACK = int(CONFIG["inverse_vol_lookback_weeks"])

sector_weekly_returns = weekly_returns[SECTOR_TICKERS].copy()

# The completed signal-week return is known at the signal observation.
# The resulting weights are not executable until the next trading session.
rolling_vol = (
    sector_weekly_returns
    .rolling(
        INV_VOL_LOOKBACK,
        min_periods=INV_VOL_LOOKBACK,
    )
    .std()
)

inv_vol_score = 1.0 / rolling_vol.replace(0.0, np.nan)

strategic_weights = inv_vol_score.div(
    inv_vol_score.sum(axis=1),
    axis=0,
)

strategic_weights = strategic_weights.dropna(how="any")

# Normalize to eliminate tiny floating-point differences.
strategic_weights = strategic_weights.div(
    strategic_weights.sum(axis=1),
    axis=0,
)

assert (strategic_weights >= 0).all().all()
assert np.allclose(
    strategic_weights.sum(axis=1),
    1.0,
    atol=1e-10,
)

STRATEGIC_WEIGHT_METHOD = "INVERSE_VOLATILITY_52W"

print("Strategic weight method:", STRATEGIC_WEIGHT_METHOD)
print("Inverse-vol lookback:", INV_VOL_LOOKBACK, "weeks")
print(
    "First complete strategic-weight signal week:",
    strategic_weights.index.min().date(),
)

display(strategic_weights.head())


Strategic weight method: INVERSE_VOLATILITY_52W
Inverse-vol lookback: 52 weeks
First complete strategic-weight signal week: 2019-06-21


Ticker,XLC,XLY,XLP,XLE,XLF,XLV,XLI,XLK,XLB,XLRE,XLU
Date,,,,,,,,,,,
2019-06-21,0.086922,0.083563,0.107224,0.071234,0.077770,0.082874,0.082420,0.080182,0.080769,0.112371,0.134671
2019-06-28,0.087578,0.084198,0.107008,0.071485,0.078118,0.083270,0.082860,0.080795,0.080811,0.110317,0.133561
2019-07-05,0.087674,0.083877,0.106359,0.071477,0.077824,0.084129,0.082927,0.080761,0.080839,0.109571,0.134561
2019-07-12,0.087605,0.083790,0.106255,0.071026,0.077773,0.083924,0.083151,0.080817,0.080599,0.109670,0.135389
2019-07-19,0.086753,0.083655,0.106500,0.070960,0.078131,0.084119,0.083195,0.080939,0.080821,0.109160,0.135767


In [ ]:
# ============================================================
# BLOCK 2.10 — DETERMINE CANONICAL RESEARCH WINDOW
# ============================================================

TSMOM_LOOKBACK_MONTHS = int(CONFIG["tsmom_lookback_months"])
TSMOM_LOOKBACK_WEEKS_APPROX = round(
    TSMOM_LOOKBACK_MONTHS * 52 / 12
)

SIGNAL_WARMUP_WEEKS = max(
    TSMOM_LOOKBACK_WEEKS_APPROX,
    int(CONFIG["slow_ema"]),
    int(CONFIG["bb_length"]) + int(CONFIG["bb_smoothing"]),
)

price_eligible_signal_week = weekly_close.index[SIGNAL_WARMUP_WEEKS]
weight_eligible_signal_week = strategic_weights.index.min()
execution_eligible_signal_week = executable_timing.index.min()

CANONICAL_SIGNAL_START = max(
    price_eligible_signal_week,
    weight_eligible_signal_week,
    execution_eligible_signal_week,
)

CANONICAL_SIGNAL_END = min(
    weekly_close.index.max(),
    strategic_weights.index.max(),
    executable_timing.index.max(),
)

CANONICAL_EXECUTION_START = executable_timing.loc[
    CANONICAL_SIGNAL_START,
    "execution_date",
]

CANONICAL_EXECUTION_END = executable_timing.loc[
    CANONICAL_SIGNAL_END,
    "execution_date",
]

print("Signal warm-up:", SIGNAL_WARMUP_WEEKS, "weeks")
print("\nCanonical signal window:")
print(
    CANONICAL_SIGNAL_START.date(),
    "to",
    CANONICAL_SIGNAL_END.date(),
)

print("\nCorresponding execution window:")
print(
    CANONICAL_EXECUTION_START.date(),
    "to",
    CANONICAL_EXECUTION_END.date(),
)


Signal warm-up: 52 weeks

Canonical signal window:
2019-06-21 to 2026-08-21

Corresponding execution window:
2019-06-24 to 2026-08-24


In [ ]:
# ============================================================
# BLOCK 2.11 — SAVE CANONICAL DATASETS
# ============================================================

daily_close_path = (
    DIRS["data_processed"] / "daily_adjusted_close_12_assets.parquet"
)

daily_open_path = (
    DIRS["data_processed"] / "daily_adjusted_open_12_assets.parquet"
)

weekly_path = (
    DIRS["data_processed"] / "weekly_signal_close_12_assets.parquet"
)

weekly_returns_path = (
    DIRS["data_processed"] / "weekly_signal_returns_12_assets.parquet"
)

weekly_timing_path = (
    DIRS["data_processed"] / "weekly_signal_execution_calendar.parquet"
)

strategic_path = (
    DIRS["data_processed"] / "weekly_signal_strategic_sector_weights.parquet"
)

coverage_path = (
    DIRS["tables"] / "market_data_coverage.csv"
)

common_daily_close.to_parquet(daily_close_path)
common_daily_open.to_parquet(daily_open_path)
weekly_close.to_parquet(weekly_path)
weekly_returns.to_parquet(weekly_returns_path)
weekly_timing.to_parquet(weekly_timing_path)
strategic_weights.to_parquet(strategic_path)
coverage.to_csv(coverage_path)

print("Saved canonical datasets:")
for p in [
    daily_close_path,
    daily_open_path,
    weekly_path,
    weekly_returns_path,
    weekly_timing_path,
    strategic_path,
    coverage_path,
]:
    print(" ", p)


Saved canonical datasets:
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/daily_adjusted_close_12_assets.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/daily_adjusted_open_12_assets.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_signal_close_12_assets.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_signal_returns_12_assets.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model

In [ ]:
# ============================================================
# BLOCK 2.12 — SAVE BLOCK 2 MANIFEST
# ============================================================

block2_manifest = {
    "project": block1["project"],
    "block": (
        "Block 2 - Universe, Data & Strategic Weight Methodology"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "timing_contract": {
        "signal_week_label_frequency": CONFIG["signal_frequency"],
        "signal_observation_config": CONFIG["signal_observation"],
        "actual_signal_observation_rule": (
            "Final available U.S. trading session in each W-FRI week"
        ),
        "rebalance_frequency": CONFIG["rebalance_frequency"],
        "nominal_rebalance_day": CONFIG["rebalance_day"],
        "execution_rule": CONFIG["rebalance_execution_rule"],
        "minimum_signal_execution_lag_sessions": int(
            CONFIG["minimum_signal_execution_lag_sessions"]
        ),
    },

    "price_source": "Yahoo Finance via yfinance",
    "price_adjustment": "auto_adjust=True; repair=True",
    "download_start": DOWNLOAD_START,
    "raw_last_date": str(raw.index.max().date()),

    "sector_tickers": SECTOR_TICKERS,
    "benchmark_ticker": BENCHMARK_TICKER,

    "common_daily_start": str(COMMON_DAILY_START.date()),
    "common_daily_end": str(COMMON_DAILY_END.date()),

    "weekly_signal_start": str(weekly_close.index.min().date()),
    "weekly_signal_end": str(weekly_close.index.max().date()),

    "strategic_weight_method": STRATEGIC_WEIGHT_METHOD,
    "inverse_vol_lookback_weeks": INV_VOL_LOOKBACK,

    "signal_warmup_weeks": SIGNAL_WARMUP_WEEKS,
    "canonical_signal_start": str(CANONICAL_SIGNAL_START.date()),
    "canonical_signal_end": str(CANONICAL_SIGNAL_END.date()),
    "canonical_execution_start": str(CANONICAL_EXECUTION_START.date()),
    "canonical_execution_end": str(CANONICAL_EXECUTION_END.date()),

    "lookahead_controls": [
        (
            "Friday-labelled weekly observations use the final real "
            "trading session in the week."
        ),
        (
            "Each signal week is mapped to the first U.S. trading "
            "session strictly after the signal observation."
        ),
        (
            "Inverse volatility uses only information available "
            "through the completed signal week."
        ),
        "ETF data before actual inception are not synthesized.",
    ],

    "saved_files": {
        "daily_adjusted_close": str(daily_close_path),
        "daily_adjusted_open": str(daily_open_path),
        "weekly_signal_close": str(weekly_path),
        "weekly_signal_returns": str(weekly_returns_path),
        "weekly_signal_execution_calendar": str(weekly_timing_path),
        "strategic_weights": str(strategic_path),
        "coverage_table": str(coverage_path),
    },
}

manifest_path = (
    DIRS["manifests"] / "block_2_universe_data_weights.json"
)

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(block2_manifest, f, indent=2)

print("Saved Block 2 manifest:")
print(manifest_path)


Saved Block 2 manifest:
/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/manifests/block_2_universe_data_weights.json


In [ ]:
# ============================================================
# BLOCK 2.13 — FINAL DIAGNOSTIC SUMMARY
# ============================================================

summary = pd.Series(
    {
        "Sector count": len(SECTOR_TICKERS),
        "Benchmark": BENCHMARK_TICKER,

        "Signal week frequency": CONFIG["signal_frequency"],
        "Configured signal observation": CONFIG["signal_observation"],
        "Nominal rebalance day": CONFIG["rebalance_day"],
        "Execution rule": CONFIG["rebalance_execution_rule"],

        "Common daily start": COMMON_DAILY_START.date(),
        "Common weekly signal start": weekly_close.index.min().date(),
        "Latest weekly signal label": weekly_close.index.max().date(),

        "Executable signal weeks": len(executable_timing),
        "Monday executions": int(
            (executable_timing["execution_weekday"] == "Monday").sum()
        ),
        "Non-Monday executions": int(
            (executable_timing["execution_weekday"] != "Monday").sum()
        ),

        "Strategic weight method": STRATEGIC_WEIGHT_METHOD,
        "Inverse-vol lookback weeks": INV_VOL_LOOKBACK,
        "Strategic weights sum to 1": bool(
            np.allclose(strategic_weights.sum(axis=1), 1.0)
        ),

        "Signal warm-up weeks": SIGNAL_WARMUP_WEEKS,
        "Canonical signal start": CANONICAL_SIGNAL_START.date(),
        "Canonical execution start": CANONICAL_EXECUTION_START.date(),
        "Canonical signal end": CANONICAL_SIGNAL_END.date(),
        "Canonical execution end": CANONICAL_EXECUTION_END.date(),
    },
    name="Block 2 status",
).to_frame()

display(summary)

print("\nBLOCK 2 COMPLETE")
print(
    "Next: Block 3 — Weekly Signal Engine: "
    "TSMOM + SuperSmoother Oscillator"
)


,Block 2 status
Sector count,11
Benchmark,SPY
Signal week frequency,W-FRI
Configured signal observation,FRIDAY_CLOSE
Nominal rebalance day,MONDAY
Execution rule,NEXT_US_TRADING_SESSION_AFTER_SIGNAL
Common daily start,2018-06-19
Common weekly signal start,2018-06-22
Latest weekly signal label,2026-08-28
Executable signal weeks,427



BLOCK 2 COMPLETE
Next: Block 3 — Weekly Signal Engine: TSMOM + SuperSmoother Oscillator
